# Chabauty–Kim : l'inégalité dimensionnelle par Iwasawa non commutative

**Source** — Sa'ar Zehavi (MPI for Mathematics), *The Chabauty–Kim dimension inequality via noncommutative Iwasawa theory*, arXiv:2611.18218v1 [math.NT], soumis le 13 novembre 2026, 61 pages.

> **Préprint — non relu.** Le texte a trois jours au moment de la distillation : rien de ce qui suit ne se lit comme un résultat acquis. On lit la source archivée, on en calcule la couche combinatoire, on cite le reste.

Copie archivée (règle `bibliography-hygiene`, identité vérifiée page 1) :
`G:\Mon Drive\MyIA\IA\Bibliographie IA\NumberTheory\2026 - Zehavi - The Chabauty-Kim dimension inequality via noncommutative Iwasawa theory (arXiv 2611.18218v1).pdf`

**Aucune ligne du PDF n'est recopiée ici.** Les énoncés sont *restitués* (notation, structure, contenu mathématique) ; la copie reste hors dépôt, dans le gisement partagé.

## Ce que l'article démontre

Pour toute courbe lisse projective géométriquement connexe C/Q de genre g ≥ 2, tout premier impair p de bonne réduction, et tout n assez grand, la conjecture de Kim (2009) est démontrée :

    dim H¹_f(G_Q,Σ, U_n^ét)  <  dim(U_n^dR / F⁰)

**Sans restriction** sur l'anneau d'endomorphismes de la jacobienne ni sur l'image de Galois du module de Tate — c'est la levée de la restriction CM de Coates–Kim. Le moteur est la théorie d'Iwasawa non commutative : le théorème 1.1 borne dim H²(G_Q,Σ, V_λ,s) par min{d_λ, (1+‖λ‖)^{D−g−1}/d_λ}, puis des annulations de sommes de caractères donnent le gain de puissance. Conséquence : les lieux de Chabauty–Kim sont finis à tout niveau assez grand, et le théorème de Mordell sur Q se retrouve inconditionnellement par cette voie.

## Ce qui est calculé, ce qui est cité

**Calculé ici** (Python stdlib seul, chaque identité vérifiée dans une sortie) :

1. la structure de racines du système C_g, et ρ = (g, g−1, …, 1) ;
2. la dimension de Weyl d_λ, par **trois routes indépendantes** ;
3. D = 2g²+g, les deux bornes du théorème 1.1, et le critère de la Remark 3.2 ;
4. les nombres de Witt et la série de Hilbert de l'algèbre de Lie libre (terme quadratique en ½) ;
5. les polynômes de Chebyshev rescalés, vérifiés numériquement ;
6. le mécanisme de parité qui donne dim L_k⁻ = b_k/2 pour k impair ;
7. le bilan arithmétique de la condition de Chabauty–Kim et le mécanisme du gain de puissance ;
8. le cas n = 1 (partie abélienne) sur un exemple documenté.

**Cité, non reproduit** : la théorie d'Iwasawa non commutative, les annulations de sommes de caractères (§4), l'estimation de second moment (Proposition 4.14), l'intégration de Coleman.

In [1]:
from fractions import Fraction
from itertools import permutations, product
from math import cos, sqrt


def rho(g: int) -> tuple:
    """rho = demi-somme des racines positives de C_g, soit (g, g-1, ..., 1)."""
    # rho est le denominateur de la formule de Weyl : chaque racine
    # positive a pour appariement <rho, alpha> = 2, 1 ou 1.
    return tuple(g - i for i in range(g))


def racines_positives(g: int) -> list:
    """Phi+ = {2e_i} u {e_i - e_j} u {e_i + e_j}, encodees par leurs coefficients."""
    # Une racine est encodee par ses coefficients sur la base (e_1, ..., e_g).
    # Un coefficient negatif porte le signe : c'est ce qui permet a la route
    # de Weyl de renvoyer une racine negative (w(2e_i) peut valoir -2e_i).
    racines = []
    for i in range(g):
        v = [0] * g
        v[i] = 2
        racines.append(tuple(v))
    for i in range(g):
        for j in range(i + 1, g):
            v = [0] * g
            v[i], v[j] = 1, -1
            racines.append(tuple(v))
            v = [0] * g
            v[i], v[j] = 1, 1
            racines.append(tuple(v))
    return racines


def appariement(mu: tuple, alpha: tuple) -> int:
    """<mu, alpha> sur la base standard (e_1, ..., e_g)."""
    # Appariement euclidien des poids et des racines : il apparait tel quel
    # dans le produit de la formule de Weyl, ou dans son developpement
    # par type de racine.
    return sum(m * a for m, a in zip(mu, alpha))


print("rho(3) =", rho(3))
print("nombre de racines positives pour g = 1, 2, 3 :",
      [len(racines_positives(g)) for g in (1, 2, 3)])
print("Phi+(C_2) =", racines_positives(2))

rho(3) = (3, 2, 1)
nombre de racines positives pour g = 1, 2, 3 : [1, 4, 9]
Phi+(C_2) = [(2, 0), (0, 2), (1, -1), (1, 1)]


### Lecture du résultat

La source ne parle pas de « la dimension de l'orbite ouverte » : **d_λ := dim_Qp V_λ,s**, où V_λ,s est la représentation irréductible de GSp_2g de poids s dont la restriction à Sp_2g est la représentation de poids dominant λ = (λ₁ ≥ … ≥ λ_g ≥ 0).

Deux conventions portent le calcul, et les deux sont explicites dans la source :

- le système de racines est celui de **C_g** : Φ⁺ = {2e_i} ∪ {e_i − e_j} ∪ {e_i + e_j} ;
- ρ, la demi-somme des racines positives, vaut **ρ = (g, g−1, …, 1)**.

Le lemme de Weyl (Fulton–Harris, Cor. 24.6) donne alors la formule produit de la Remark 3.2, que l'on calcule ci-dessous en arithmétique exacte :

    d_λ = ∏_{α ∈ Φ⁺} ( 1 + ⟨λ, α⟩ / ⟨ρ, α⟩ )

In [2]:
def d_lambda_produit(g: int, lam: tuple) -> Fraction:
    """Route 1 -- le produit sur les racines positives (Remark 3.2 de la source)."""
    r = rho(g)
    total = Fraction(1)
    for alpha in racines_positives(g):
        total *= 1 + Fraction(appariement(lam, alpha), appariement(r, alpha))
    return total


exemples = [(1, (0,)), (1, (1,)), (1, (5,)), (2, (0, 0)), (2, (1, 0)),
             (2, (1, 1)), (2, (2, 2)), (3, (1, 0, 0)), (3, (1, 1, 1))]
for g, lam in exemples:
    print(f"g={g}  lambda={lam}  d_lambda={d_lambda_produit(g, lam)}")

g=1  lambda=(0,)  d_lambda=1
g=1  lambda=(1,)  d_lambda=2
g=1  lambda=(5,)  d_lambda=6
g=2  lambda=(0, 0)  d_lambda=1
g=2  lambda=(1, 0)  d_lambda=4
g=2  lambda=(1, 1)  d_lambda=5
g=2  lambda=(2, 2)  d_lambda=14
g=3  lambda=(1, 0, 0)  d_lambda=6
g=3  lambda=(1, 1, 1)  d_lambda=14


In [3]:
def d_lambda_ferme(g: int, lam: tuple) -> Fraction:
    """Route 2 -- la meme formule, cas par cas : 2e_i, e_i - e_j, e_i + e_j."""
    r = rho(g)
    # Le produit se factorise par famille de racines : chaque famille a
    # son propre terme (1 + <lambda, alpha>/<rho, alpha>).
    p_2e = Fraction(1)
    for i in range(g):
        p_2e *= 1 + Fraction(2 * lam[i], 2 * r[i])
    p_moins = Fraction(1)
    p_plus = Fraction(1)
    for i in range(g):
        for j in range(i + 1, g):
            p_moins *= 1 + Fraction(lam[i] - lam[j], r[i] - r[j])
            p_plus *= 1 + Fraction(lam[i] + lam[j], r[i] + r[j])
    return p_2e * p_moins * p_plus


# Lot de poids dominants sur lequel les deux routes sont comparees :
# poids d'axe (n, 0, ...), poids diagonaux (n, n, ...) et poids mixtes.
poids = []
for g in (1, 2, 3):
    poids += [(g, (n,) + (0,) * (g - 1)) for n in range(4)]
    poids += [(g, tuple([n] * g)) for n in range(3)]
    if g >= 2:
        poids += [(g, (n, m) + (0,) * (g - 2)) for n in (0, 1, 2) for m in (0, 1)]

# Une seule divergence ferait passer le carnet a cote de l'article :
# on compte les ecarts, on ne se contente pas d'afficher un accord.
ecarts = [(g, lam) for g, lam in poids
          if d_lambda_produit(g, lam) != d_lambda_ferme(g, lam)]
print(f"{len(poids)} poids dominants compares, ecarts entre les deux routes : {len(ecarts)}")
print("tableau (route 2, formule close) :")
for g in (1, 2, 3):
    vals = [(lam, d_lambda_ferme(g, lam)) for lam in [(n,) + (0,) * (g - 1) for n in range(4)]]
    print(f"  g={g} :", ", ".join(f"{lam} -> {d}" for lam, d in vals))

33 poids dominants compares, ecarts entre les deux routes : 0
tableau (route 2, formule close) :
  g=1 : (0,) -> 1, (1,) -> 2, (2,) -> 3, (3,) -> 4
  g=2 : (0, 0) -> 1, (1, 0) -> 4, (2, 0) -> 10, (3, 0) -> 20
  g=3 : (0, 0, 0) -> 1, (1, 0, 0) -> 6, (2, 0, 0) -> 21, (3, 0, 0) -> 56


In [4]:
def determinant(matrice: list) -> Fraction:
    """Determinant par elimination de Gauss, en arithmetique exacte."""
    n = len(matrice)
    M = [list(ligne) for ligne in matrice]
    det = Fraction(1)
    for col in range(n):
        pivot = None
        for row in range(col, n):
            if M[row][col] != 0:
                pivot = row
                break
        if pivot is None:
            return Fraction(0)
        if pivot != col:
            M[col], M[pivot] = M[pivot], M[col]
            det = -det
        det *= M[col][col]
        for row in range(col + 1, n):
            if M[row][col] != 0:
                facteur = M[row][col] / M[col][col]
                for k in range(col, n):
                    M[row][k] -= facteur * M[col][k]
    return det


def groupe_weyl(g: int) -> list:
    """Le groupe de Weyl de C_g : permutations signees de la base standard."""
    # g! * 2^g elements : la sommation de la route 3 est exhaustive,
    # elle n'echantillonne pas le groupe.
    return [(perm, signes)
            for perm in permutations(range(g))
            for signes in product((1, -1), repeat=g)]


def image_racine(alpha: tuple, perm: tuple, signes: tuple) -> tuple:
    """Image de la racine alpha par w, avec w(e_k) = signes[k] e_{perm[k]}."""
    g = len(alpha)
    # w agit lineairement sur la base : la somme des coefficients signes
    # encode a la fois la permutation et le signe de la racine imagee.
    image = [0] * g
    for k in range(g):
        image[perm[k]] += alpha[k] * signes[k]
    return tuple(image)


def d_lambda_weyl(g: int, lam: tuple) -> Fraction:
    """Route 3 -- la sommation sur le groupe de Weyl."""
    r = rho(g)
    # La formule de Weyl s'ecrit avec lambda + rho, et non lambda seul.
    lam_rho = tuple(lam[i] + r[i] for i in range(g))
    total = Fraction(0)
    for perm, signes in groupe_weyl(g):
        # M est la matrice de w sur la base (e_1, ..., e_g) : son determinant
        # est le signe de w, qui apparait dans la somme.
        M = [[0] * g for _ in range(g)]
        for k in range(g):
            M[perm[k]][k] = signes[k]
        produit = Fraction(1)
        for alpha in racines_positives(g):
            num = appariement(lam_rho, image_racine(alpha, perm, signes))
            produit *= Fraction(num, appariement(r, alpha))
        total += determinant(M) * produit
    # La somme est divisee par |W| : c'est la moyenne de Weyl.
    return total / len(groupe_weyl(g))


# Les trois routes doivent coindre sur tous les poids du lot : la route 3
# verifie le plus, car elle est la formule de Weyl elle-meme.
ecarts = [(g, lam) for g, lam in poids
          if not (d_lambda_produit(g, lam) == d_lambda_ferme(g, lam) == d_lambda_weyl(g, lam))]
print(f"{len(poids)} poids dominants, trois routes independantes, ecarts : {len(ecarts)}")
print("Sp_4 (g=2) :", [(lam, d_lambda_weyl(2, lam))
                       for lam in ((0, 0), (1, 0), (1, 1), (2, 0), (2, 1), (2, 2))])
print("Sp_6 (g=3) :", [(lam, d_lambda_weyl(3, lam))
                       for lam in ((0, 0, 0), (1, 0, 0), (1, 1, 0), (1, 1, 1))])

33 poids dominants, trois routes independantes, ecarts : 0
Sp_4 (g=2) : [((0, 0), Fraction(1, 1)), ((1, 0), Fraction(4, 1)), ((1, 1), Fraction(5, 1)), ((2, 0), Fraction(10, 1)), ((2, 1), Fraction(16, 1)), ((2, 2), Fraction(14, 1))]
Sp_6 (g=3) : [((0, 0, 0), Fraction(1, 1)), ((1, 0, 0), Fraction(6, 1)), ((1, 1, 0), Fraction(14, 1)), ((1, 1, 1), Fraction(14, 1))]


### Trois routes, une même valeur

La formule produit de la Remark 3.2 a été obtenue « par analyse cas par cas des trois types de racines ». Ce cas par cas n'est pas une approximation : c'est un regroupement de facteurs, et il se calcule. On a donc implémenté trois routes **indépendantes** :

1. le **produit** sur les racines positives (la formule de la source) ;
2. la **formule close**, où les trois types de racines sont traités séparément (le « cas par cas » de la source) ;
3. la **sommation sur le groupe de Weyl**, forme définitionnelle de la formule de Weyl (ici les racines sont représentées par leurs coefficients, un coefficient négatif portant le signe).

Les trois coïncident sur l'échantillon. C'est un **contrôle de cohérence**, pas une preuve : elles pourraient être trois écritures du même calcul.

In [5]:
def dim_sp(g: int) -> int:
    """D = dim Sp_2g = 2g^2 + g."""
    # D entre dans l'exposant D - g - 1 de la borne non triviale : il
    # suffit de le mal evaluer pour decaler le regime de la borne.
    return 2 * g * g + g


def norme_lambda(lam: tuple) -> float:
    """||lambda||_2 = (somme des lambda_i^2)^{1/2}."""
    # La borne du theoreme 1.1 ne depend du poids que par cette norme :
    # deux poids de meme norme ont la meme borne.
    return sum(x * x for x in lam) ** 0.5


def bornes(g: int, lam: tuple) -> dict:
    """Les deux bornes du theoreme 1.1, et le critere de la Remark 3.2."""
    d = d_lambda_produit(g, lam)
    D = dim_sp(g)
    nrm = norme_lambda(lam)
    # Borne triviale : la dimension de la representation elle-meme.
    triviale = float(d)
    # Borne non triviale : elle oppose la norme du poids a d_lambda.
    non_triviale = (1 + nrm) ** (D - g - 1) / float(d)
    # Seuil de la Rem. 3.2 : d_lambda > (1+||lambda||)^((D-g-1)/2).
    seuil = (1 + nrm) ** ((D - g - 1) / 2)
    return {"d": d, "triviale": triviale, "non_triviale": non_triviale,
            "borne": min(triviale, non_triviale),
            "critere": float(d) > seuil, "seuil": seuil}


print("D = dim Sp_2g :", {g: dim_sp(g) for g in (1, 2, 3)})
print()
print("g=1 (Sp_2, D=3) -- la borne non triviale l'emporte des lambda != 0")
for lam in [(0,), (1,), (2,), (5,)]:
    b = bornes(1, lam)
    print(f"   lambda={lam}  d={b['d']}  borne={b['borne']:.4g}  critere Rem. 3.2 = {b['critere']}")
print()
print("g=2 (Sp_4, D=10) -- table des bornes")
print(f"{'lambda':<14}{'d':>12}{'||lambda||':>12}{'triviale':>12}{'non triviale':>16}{'borne':>14}  Rem. 3.2")
# Le tableau balaie des poids d'axe, des poids diagonaux et des poids de
# direction generique : le critere depend de la direction, pas seulement
# de la norme -- deux poids de meme norme peuvent repondre differemment.
for lam in [(0, 0), (1, 0), (1, 1), (5, 0), (5, 5), (20, 20), (50, 50),
            (100, 100), (200, 100), (600, 300), (2000, 1000)]:
    b = bornes(2, lam)
    print(f"{str(lam):<14}{str(b['d']):>12}{norme_lambda(lam):>12.4f}{b['triviale']:>12.4g}"
          f"{b['non_triviale']:>16.4g}{b['borne']:>14.4g}  {'oui' if b['critere'] else 'non'}")

D = dim Sp_2g : {1: 3, 2: 10, 3: 21}

g=1 (Sp_2, D=3) -- la borne non triviale l'emporte des lambda != 0
   lambda=(0,)  d=1  borne=1  critere Rem. 3.2 = False
   lambda=(1,)  d=2  borne=1  critere Rem. 3.2 = True
   lambda=(2,)  d=3  borne=1  critere Rem. 3.2 = True
   lambda=(5,)  d=6  borne=1  critere Rem. 3.2 = True

g=2 (Sp_4, D=10) -- table des bornes
lambda                   d  ||lambda||    triviale    non triviale         borne  Rem. 3.2
(0, 0)                   1      0.0000           1               1             1  non
(1, 0)                   4      1.0000           4              32             4  non
(1, 1)                   5      1.4142           5            95.6             5  non
(5, 0)                  56      5.0000          56            4999            56  non
(5, 5)                  91      7.0711          91       2.452e+04            91  non
(20, 20)              3311     28.2843        3311       5.578e+06          3311  non
(50, 50)             45526     70

### Pourquoi un minimum de deux bornes

Le théorème 1.1 est un **amalgame de deux bornes** :

- la borne **triviale**, dim H²(G, V_λ,s) ≪ d_λ, valable pour tout poids ;
- la borne **non triviale** (théorème 3.1), dim Ker η ≪ (1+‖λ‖)^{D−g−1} / d_λ, valable **hors d'un ensemble fini de poids** E.

La Remark 3.2 dit quand la seconde l'emporte sur la première :

    d_λ > (1+‖λ‖)^{(D−g−1)/2}  =  (1+‖λ‖)^{g²−1/2}      (D = 2g²+g)

Trois lectures de la table :

- pour **g = 1** (Sp_2), la borne non triviale l'emporte dès que λ ≠ 0 ;
- pour **g = 2** (Sp_4), sur l'axe (n, 0) et sur la diagonale (n, n), d_λ croît comme ‖λ‖³ — moins vite que le seuil ‖λ‖^{g²−1/2} = ‖λ‖^{3,5} : le critère **ne** se vérifie jamais, quelle que soit la taille de λ ;
- mais dans une **direction générique** (les deux coordonnées du même ordre et distinctes, ici λ₂ = λ₁/2), les quatre facteurs de la formule de Weyl contribuent et d_λ croît comme ‖λ‖⁴ : le critère est alors vérifié pour ‖λ‖ assez grand.

Autrement dit, le seuil ne dépend pas seulement de la **norme** de λ, mais de sa **direction** dans l'espace des poids. C'est un point que la seule table des valeurs classiques ne montre pas.

In [6]:
def mu_mobius(n: int) -> int:
    """Fonction de Mobius : 0 si n a un facteur carre, sinon (-1) au nombre de facteurs premiers distincts."""
    if n < 1:
        raise ValueError("n doit etre un entier positif")
    # Un facteur carre rend mu nul : c'est ce qui borne la somme de Witt.
    reste, parite, p = n, 1, 2
    while p * p <= reste:
        if reste % p == 0:
            reste //= p
            parite = -parite
            if reste% p == 0:
                return 0
        p += 1 if p == 2 else 2
    if reste > 1:
        parite = -parite
    return parite


def diviseurs(n: int) -> list:
    """Diviseurs positifs de n, dans l'ordre croissant."""
    return [d for d in range(1, n + 1) if n % d == 0]


def witt_nombre(m: int, n: int) -> int:
    """Dimension de la piece de degre n de l'algebre de Lie libre sur m generateurs."""
    # Forme close de Witt : (1/n) SOMME_{d|n} mu(d) m^(n/d).
    total = sum(mu_mobius(d) * m ** (n // d) for d in diviseurs(n))
    return int(Fraction(total, n))


def log_serie(coeffs: list) -> list:
    """Logarithme de la serie 1 + a_1 t + ..., tronque a l'ordre len(coeffs) - 1.
    Recurrence : b_k = a_k - (1/k) SOMME_{i<k} i b_i a_{k-i}."""
    n = len(coeffs) - 1
    b = [Fraction(0)] * (n + 1)
    for k in range(1, n + 1):
        # Attention : le terme a_k n'est PAS divise par k.
        s = k * Fraction(coeffs[k])
        for i in range(1, k):
            s -= i * b[i] * coeffs[k - i]
        b[k] = s / k
    return b


def serie_hilbert_libre(m: int, N: int) -> list:
    """Serie de Hilbert de l'algebre de Lie libre, par la forme logarithmique."""
    # H(t) = -SOMME_{d >= 1} (mu(d)/d) log(1 - m t^d).
    H = [Fraction(0)] * (N + 1)
    for d in range(1, N + 1):
        if mu_mobius(d) == 0:
            continue
        # Chaque terme du logarithme est calcule sur 1 - m t^d.
        A = [Fraction(0)] * (N + 1)
        A[0] = Fraction(1)
        A[d] = Fraction(-m)
        L = log_serie(A)
        for n in range(1, N + 1):
            H[n] -= Fraction(mu_mobius(d), d) * L[n]
    return H


print("mu :", {n: mu_mobius(n) for n in (1, 2, 3, 4, 6, 12, 30, 36)})
print()
for m in (2, 4, 6):
    H = serie_hilbert_libre(m, 8)
    w = [witt_nombre(m, n) for n in range(1, 9)]
    # Les deux formes doivent coindre terme a terme : c'est le controle de
    # vraisemblance de la couche executee (une forme seule ne prouve rien).
    identique = all(H[n] == w[n - 1] for n in range(1, 9))
    print(f"m={m} : Hilbert (forme logarithmique) = {[str(H[n]) for n in range(1, 5)]}")
    print(f"      Witt (forme close)                = {w[:4]}   identique : {identique}")
print()
print("coefficient quadratique (m^2 - m)/2 -- la moitie qui structure la suite :")
for m in (2, 4, 6):
    H = serie_hilbert_libre(m, 4)
    print(f"   m={m} : coefficient t^2 = {H[2]}   (m^2 - m)/2 = {Fraction(m * m - m, 2)}")

mu : {1: 1, 2: -1, 3: -1, 4: 0, 6: 1, 12: 0, 30: -1, 36: 0}

m=2 : Hilbert (forme logarithmique) = ['2', '1', '2', '3']
      Witt (forme close)                = [2, 1, 2, 3]   identique : True
m=4 : Hilbert (forme logarithmique) = ['4', '6', '20', '60']
      Witt (forme close)                = [4, 6, 20, 60]   identique : True
m=6 : Hilbert (forme logarithmique) = ['6', '15', '70', '315']
      Witt (forme close)                = [6, 15, 70, 315]   identique : True

coefficient quadratique (m^2 - m)/2 -- la moitie qui structure la suite :
   m=2 : coefficient t^2 = 1   (m^2 - m)/2 = 1
   m=4 : coefficient t^2 = 6   (m^2 - m)/2 = 6
   m=6 : coefficient t^2 = 15   (m^2 - m)/2 = 15


### De la majoration à la croissance

L'algèbre de Lie graduée de la source est notée L, et **L = 𝔏(V)/(ρ)** où 𝔏(V) est l'algèbre de Lie **libre** sur V = L₁ = H¹_et(C̄; Q_p), de dimension 2g. L est donc un **quotient** de l'algèbre de Lie libre : la dimension b_k de sa pièce de degré k est majorée par le nombre de Witt sur 2g générateurs — c'est ce majorant que l'on calcule ici.

La série de Hilbert de l'algèbre de Lie libre s'écrit

    H(t) = − Σ_{d ≥ 1} μ(d)/d · log(1 − m t^d),      m = 2g,

et son coefficient de degré n est exactement le nombre de Witt (1/n) Σ_{d|n} μ(d) m^{n/d}. Les deux écritures sont confrontées ci-dessus, terme à terme.

Le terme de degré 2 vaut (m² − m)/2 : c'est le premier endroit où un **½** apparaît, et il n'est pas fortuit — il vient de la relation d'anticommutation des générateurs, dont le terme quadratique est (1/2)[x, y].

In [7]:
def chebyshev_coefs(d: int) -> list:
    """Coefficients entiers du polynome de Chebyshev rescale P_d (P_0 = 2, P_1 = x)."""
    if d == 0:
        return [2]
    if d == 1:
        return [0, 1]
    # Recurrence P_d = x P_{d-1} - P_{d-2} : elle est codee par
    # substitution de polynomes (multiplication par x, puis soustraction).
    P0, P1 = [2], [0, 1]
    for _ in range(2, d + 1):
        suivant = [0] * (len(P1) + 1)
        for i, c in enumerate(P1):
            suivant[i + 1] += c
        for i, c in enumerate(P0):
            suivant[i] -= c
        P0, P1 = P1, suivant
    return P1


def chebyshev_rescale(d: int, x) -> Fraction:
    """P_d(x), evalue exactement."""
    # L'evaluation reste exacte : les coefficients sont entiers et x est
    # rationnel, donc tout le calcul se fait en Fraction.
    total = Fraction(0)
    for i, c in enumerate(chebyshev_coefs(d)):
        total += c * Fraction(x) ** i
    return total


def polynome_texte(coefs: list) -> str:
    """Ecriture lisible d'un polynome a coefficients entiers."""
    morceaux = []
    for i, c in enumerate(coefs):
        if c == 0:
            continue
        signe = "-" if c < 0 else ("+" if morceaux else "")
        valeur = abs(c)
        if i == 0:
            terme = str(valeur)
        elif i == 1:
            terme = "x" if valeur == 1 else f"{valeur}x"
        else:
            terme = f"x^{i}" if valeur == 1 else f"{valeur}x^{i}"
        morceaux.append(signe + terme)
    return " ".join(morceaux) if morceaux else "0"


print("premiers polynomes de Chebyshev rescale :")
for d in range(1, 6):
    print(f"   P_{d}(x) = {polynome_texte(chebyshev_coefs(d))}")
print()
print("identite P_d(2 cos theta) = 2 cos(d theta) :")
for theta in (0.3, 1.1, 2.0):
    ecarts = []
    for d in (1, 2, 3, 5, 8):
        x = 2 * cos(theta)
        ecarts.append(abs(float(chebyshev_rescale(d, x)) - 2 * cos(d * theta)))
    print(f"   theta = {theta} : ecart maximal = {max(ecarts):.3e}")
print()
print("forme close ((x +- racine(x^2 - 4))/2)^d + ... :")
for x in (2.5, 3.0, 5.0):
    # La forme close est celle que la formule de Filip fait apparaitre :
    # c'est elle qu'on compare a la recurrence.
    recurrence = float(chebyshev_rescale(6, x))
    racine = sqrt(x * x - 4)
    forme = ((x + racine) / 2) ** 6 + ((x - racine) / 2) ** 6
    print(f"   x = {x} : recurrence = {recurrence:.10f}   forme close = {forme:.10f}")

premiers polynomes de Chebyshev rescale :
   P_1(x) = x
   P_2(x) = -2 +x^2
   P_3(x) = -3x +x^3
   P_4(x) = 2 -4x^2 +x^4
   P_5(x) = 5x -5x^3 +x^5

identite P_d(2 cos theta) = 2 cos(d theta) :
   theta = 0.3 : ecart maximal = 1.554e-15
   theta = 1.1 : ecart maximal = 6.661e-16
   theta = 2.0 : ecart maximal = 2.220e-16

forme close ((x +- racine(x^2 - 4))/2)^d + ... :
   x = 2.5 : recurrence = 64.0156250000   forme close = 64.0156250000
   x = 3.0 : recurrence = 322.0000000000   forme close = 322.0000000000
   x = 5.0 : recurrence = 12098.0000000000   forme close = 12098.0000000000


### La formule du caractère (Filip)

La source passe ensuite en groupe compact réel, USp(2g), et y utilise la formule explicite du caractère de la pièce de degré n de L, due à Filip :

    χ_n(u) = (1/n) Σ_{d|n} μ(n/d) · P_d( χ_V(u^{n/d}) ),

où P_d est le **polynôme de Chebyshev rescalé**

    P_d(x) = ( (x + √(x²−4))/2 )^d + ( (x − √(x²−4))/2 )^d,

et χ_V le caractère de la représentation standard de USp(2g).

C'est cette formule qui alimente les deux estimations que la source utilise ensuite :

- **Proposition 4.9** : m_n(λ) ≪ d_λ · b_n · n^{−D/2} — la multiplicité d'un poids dominant dans L_n est rare ;
- **Proposition 4.14** : Σ_λ m_n(λ) · d_λ · ‖λ‖² ≪ b_n · n — l'estimation du second moment.

Ces deux estimations sont **citées, non reproduites** : elles demandent l'orthogonalité des caractères sur USp(2g) et une analyse de sommes de caractères. Le polynôme de Chebyshev, en revanche, se calcule exactement — c'est l'objet de la cellule précédente.

In [8]:
print("Le mecanisme du 1/2 : la trace de la conjugaison complexe")
print()
print("Sur les pieces de degre n de l'algebre enveloppante, la source etablit la recurrence")
print("   tr(sigma | A_n) = tr(sigma | A_{n-2}),   avec   tr(sigma | A_0) = 1   et   tr(sigma | V) = 0")
print("d'ou une suite de traces nulle sur les degres impairs :")
# La recurrence a pas 2 et la valeur initiale 1 : la trace est nulle sur
# les degres impairs, nulle part ailleurs. C'est l'origine du facteur 1/2.
traces = [1 if n % 2 == 0 else 0 for n in range(8)]
print("   n        :", list(range(8)))
print("   tr(sigma):", traces)
print()
print("Consequence (Remark 4.17) : pour k impair, dim L_k^- = (1/2) dim L_k = b_k / 2 ;")
print("en general (Proposition 4.16), dim L_k^- = b_k/2 + O(alpha^(n/2)/n).")
print()
print("Sur le modele de l'algebre de Lie libre (majorant de b_k), la parite se verifie :")
for g in (1, 2, 3):
    m = 2 * g
    impair = [(k, witt_nombre(m, k)) for k in range(1, 13, 2)]
    # b_k pair pour k impair : le modele reproduit donc bien la parite que
    # la trace impose dans la source.
    tous_pairs = all(b % 2 == 0 for _, b in impair)
    print(f"   g={g} (m={m}) : b_k pour k impair = {impair}")
    print(f"        tous pairs : {tous_pairs}   ->  b_k / 2 = {[b // 2 for _, b in impair]}")

Le mecanisme du 1/2 : la trace de la conjugaison complexe

Sur les pieces de degre n de l'algebre enveloppante, la source etablit la recurrence
   tr(sigma | A_n) = tr(sigma | A_{n-2}),   avec   tr(sigma | A_0) = 1   et   tr(sigma | V) = 0
d'ou une suite de traces nulle sur les degres impairs :
   n        : [0, 1, 2, 3, 4, 5, 6, 7]
   tr(sigma): [1, 0, 1, 0, 1, 0, 1, 0]

Consequence (Remark 4.17) : pour k impair, dim L_k^- = (1/2) dim L_k = b_k / 2 ;
en general (Proposition 4.16), dim L_k^- = b_k/2 + O(alpha^(n/2)/n).

Sur le modele de l'algebre de Lie libre (majorant de b_k), la parite se verifie :
   g=1 (m=2) : b_k pour k impair = [(1, 2), (3, 2), (5, 6), (7, 18), (9, 56), (11, 186)]
        tous pairs : True   ->  b_k / 2 = [1, 1, 3, 9, 28, 93]
   g=2 (m=4) : b_k pour k impair = [(1, 4), (3, 20), (5, 204), (7, 2340), (9, 29120), (11, 381300)]
        tous pairs : True   ->  b_k / 2 = [2, 10, 102, 1170, 14560, 190650]
   g=3 (m=6) : b_k pour k impair = [(1, 6), (3, 70), (5, 1554), 

### L'architecture de la preuve : pourquoi borner H² suffit

Trois pièces s'imbriquent.

**1. La dimension locale** (Proposition 4.15). La comparaison étale/de Rham d'Olsson donne dim gr_k U = b_k, donc dim U_n = Σ_{k≤n} b_k ; Kim majore dim F⁰gr_k U par g^k, d'où

    dim(U_n^dR / F⁰)  ≫  Σ_{k≤n} b_k − g^n.

C'est le membre de **droite** de la condition de Chabauty–Kim.

**2. Le membre de gauche** (équation 44). La suite exacte de la série centrale descendante donne

    dim H¹_f(G, U_n^ét)  ≤  Σ_{k≤n} dim H²(G, L_k)  +  Σ_{k≤n} dim L_k⁻,

où L_k⁻ est le (−1)-espace propre de la conjugaison complexe sur L_k.

**3. Le ½** (Proposition 4.16 et Remark 4.17). Pour k impair, dim L_k⁻ = (1/2) dim L_k = b_k/2 ; en général dim L_k⁻ = b_k/2 + O(α^{n/2}/n).

La condition de Kim se réécrit alors

    Σ_{k≤n} dim H²(G, L_k)   <   (1/2) Σ_{k≤n} b_k  −  g^n,

et le terme g^n est négligeable devant la croissance exponentielle de b_k. **Toute la démonstration tient donc dans la majoration des H²** — et c'est là que le théorème 1.4 intervient :

    dim H²(G, L_n)  <  A · b_n · n^{−1/(D+1)}.

La cellule suivante met en scène ce mécanisme, avec un majorant de b_k et une constante A illustrative (la constante de la source n'est pas effective).

In [9]:
from math import log10


def somme_bornes(g: int, n: int) -> tuple:
    """Modele : b_k = nombre de Witt sur 2g generateurs ; poids min(1, k^{-1/(D+1)}).
    Les sommes sont normalisees par b_n : elles tiennent alors dans les flottants,
    la somme exacte etant trop grande pour etre convertie en chaine."""
    m = 2 * g
    D = dim_sp(g)
    b = [witt_nombre(m, k) for k in range(1, n + 1)]
    exact = sum(b)
    # Reference de normalisation : b_n. La somme exacte depasse la limite de
    # conversion en chaine de Python (4300 chiffres), on ne rapporte donc
    # que son ordre de grandeur, en base 10.
    reference = b[-1]
    total = sum(x / reference for x in b)
    # La borne triviale sert pour les petits k, la borne avec gain de
    # puissance pour les grands : d'ou la ponderation min(1, k^{-1/(D+1)}).
    pondere = sum((b[k - 1] / reference) * min(1.0, k ** (-1.0 / (D + 1)))
                  for k in range(1, n + 1))
    return total, pondere, log10(exact)


print("Le gain de puissance du theoreme 1.4 : dim H^2(G, L_k) < A b_k k^{-1/(D+1)}")
print("Sommee sur k <= n : la borne triviale sert pour les petits k, la borne avec gain pour les grands.")
print("Modele A = 1 (la constante A de la source n'est pas effective), b_k = nombre de Witt :")
print("Les sommes sont normalisees par b_n (les valeurs exactes depassent la limite de conversion).")
print()
for g in (2, 3):
    D = dim_sp(g)
    print(f"-- g={g} (Sp_{2 * g}, D={D})")
    for n in (100, 1000, 10000):
        total, pondere, log_exact = somme_bornes(g, n)
        print(f"   n={n:>6} : somme b_k / b_n = {total:.6f}   [log10 de la somme exacte = {log_exact:.1f}]")
        print(f"              moitie de la somme = {total / 2:.6f}")
        # Un rapport inferieur a 1 signifie que la somme ponderee passe
        # sous la moitie de la somme triviale : c'est l'economie de puissance.
        print(f"              somme ponderee = {pondere:.6f}   rapport a la moitie = {pondere / (total / 2):.4f}")
    # Le seuil theorique 2^(D+1) annonce le regime ou le gain l'emporte.
    print(f"   ordre de grandeur du seuil : 2^(D+1) = {2 ** (D + 1)}")
    print()

Le gain de puissance du theoreme 1.4 : dim H^2(G, L_k) < A b_k k^{-1/(D+1)}
Sommee sur k <= n : la borne triviale sert pour les petits k, la borne avec gain pour les grands.
Modele A = 1 (la constante A de la source n'est pas effective), b_k = nombre de Witt :
Les sommes sont normalisees par b_n (les valeurs exactes depassent la limite de conversion).

-- g=2 (Sp_4, D=10)
   n=   100 : somme b_k / b_n = 1.337854   [log10 de la somme exacte = 58.3]
              moitie de la somme = 0.668927
              somme ponderee = 0.880491   rapport a la moitie = 1.3163
   n=  1000 : somme b_k / b_n = 1.333779   [log10 de la somme exacte = 599.2]
              moitie de la somme = 0.666889
              somme ponderee = 0.711819   rapport a la moitie = 1.0674


   n= 10000 : somme b_k / b_n = 1.333378   [log10 de la somme exacte = 6016.7]
              moitie de la somme = 0.666689
              somme ponderee = 0.577189   rapport a la moitie = 0.8658
   ordre de grandeur du seuil : 2^(D+1) = 2048

-- g=3 (Sp_6, D=21)
   n=   100 : somme b_k / b_n = 1.202434   [log10 de la somme exacte = 75.9]
              moitie de la somme = 0.601217
              somme ponderee = 0.975422   rapport a la moitie = 1.6224
   n=  1000 : somme b_k / b_n = 1.200240   [log10 de la somme exacte = 775.2]
              moitie de la somme = 0.600120
              somme ponderee = 0.876816   rapport a la moitie = 1.4611


   n= 10000 : somme b_k / b_n = 1.200024   [log10 de la somme exacte = 7777.6]
              moitie de la somme = 0.600012
              somme ponderee = 0.789536   rapport a la moitie = 1.3159
   ordre de grandeur du seuil : 2^(D+1) = 4194304



### Le cas n = 1 : Chabauty classique

À la première troncature, U₁^ét est la partie **abélienne** : V = T_p(J) ⊗ Q_p, de dimension 2g, et dim H¹(G, V) est le rang de Mordell–Weil de la jacobienne. La condition du diagramme de Kim à ce niveau s'écrit r < dim(U₁^dR / F⁰), que la Proposition 4.15 majore par b₁ − g = 2g − g. La condition classique de Chabauty (Coleman) est plus exigeante encore, r < g.

L'exemple de référence est la **cubique de Selmer** 3x³ + 4y³ + 5z³ = 0 : elle a des points sur chaque Q_p et sur R, mais **aucun point rationnel** — le contre-exemple fondateur du principe local-global, déjà computation dans le dépôt ([Serre100/24](../Serre100/24-hasse-minkowski-principe-local-global.ipynb)). On en compte ci-dessous les points sur les petits corps finis, et on repère les singularités de réduction (la cubique est singulière en 5, ce qui est cohérent avec l'hypothèse de bonne réduction hors d'un ensemble fini de places).

In [10]:
print("Le cas n = 1 : la partie abelienne")
for g in (1, 2, 3):
    b1 = witt_nombre(2 * g, 1)
    # Au niveau n = 1, la theorie se reduit a la theorie de Chabauty
    # classique : comparer les deux conditions est instructif.
    print(f"   genre g={g} : dim V = dim H^1_et(J) = 2g = {2 * g} ; "
          f"dim(U_1^dR / F^0) ≫ b_1 - g = {b1} - {g}")
    print(f"        condition du diagramme de Kim : r < {b1 - g} ; "
          f"condition de Chabauty classique : r < g = {g}")
print()


def compter_points_selmer(p: int) -> int:
    """Nombre de points projectifs F_p de la cubique de Selmer 3x^3 + 4y^3 + 5z^3 = 0."""
    affines = 0
    # On compte en affines puis on projectivise : chaque point projectif
    # non nul a p - 1 representatives affines (le point (0,0,0) en a un).
    for x in range(p):
        for y in range(p):
            for z in range(p):
                if (3 * x ** 3 + 4 * y ** 3 + 5 * z ** 3) % p == 0:
                    affines += 1
    return (affines - 1) // (p - 1)


def compter_singuliers(p: int) -> int:
    """Nombre de points projectifs singuliers de la cubique de Selmer modulo p."""
    affines = 0
    for x in range(p):
        for y in range(p):
            for z in range(p):
                if (3 * x ** 3 + 4 * y ** 3 + 5 * z ** 3) % p != 0:
                    continue
                # Un point est singulier quand les trois derivees partielles
                # s'annulent : ici 9x^2, 12y^2 et 15z^2.
                if (9 * x * x) % p == 0 and (12 * y * y) % p == 0 and (15 * z * z) % p == 0:
                    affines += 1
    return (affines - 1) // (p - 1)


print("La cubique de Selmer 3x^3 + 4y^3 + 5z^3 = 0 (genre 1) :")
for p in (5, 7, 11, 13, 17, 19):
    print(f"   p={p} : #C(F_p) = {compter_points_selmer(p):>4}   "
          f"points singuliers = {compter_singuliers(p)}")
print()
print("Rang cite : le jacobien de la cubique de Selmer est de rang 0 sur Q (fait classique, cite).")
print("La cubique n'a aucun point rationnel (Selmer 1951) malgre des points partout localement :")
print("c'est le contre-exemple fondateur du principe local-global. La condition de Chabauty")
print("est donc satisfaite des le niveau n = 1.")

Le cas n = 1 : la partie abelienne
   genre g=1 : dim V = dim H^1_et(J) = 2g = 2 ; dim(U_1^dR / F^0) ≫ b_1 - g = 2 - 1
        condition du diagramme de Kim : r < 1 ; condition de Chabauty classique : r < g = 1
   genre g=2 : dim V = dim H^1_et(J) = 2g = 4 ; dim(U_1^dR / F^0) ≫ b_1 - g = 4 - 2
        condition du diagramme de Kim : r < 2 ; condition de Chabauty classique : r < g = 2
   genre g=3 : dim V = dim H^1_et(J) = 2g = 6 ; dim(U_1^dR / F^0) ≫ b_1 - g = 6 - 3
        condition du diagramme de Kim : r < 3 ; condition de Chabauty classique : r < g = 3

La cubique de Selmer 3x^3 + 4y^3 + 5z^3 = 0 (genre 1) :
   p=5 : #C(F_p) =    6   points singuliers = 1
   p=7 : #C(F_p) =    3   points singuliers = 0
   p=11 : #C(F_p) =   12   points singuliers = 0
   p=13 : #C(F_p) =    9   points singuliers = 0
   p=17 : #C(F_p) =   18   points singuliers = 0
   p=19 : #C(F_p) =   12   points singuliers = 0

Rang cite : le jacobien de la cubique de Selmer est de rang 0 sur Q (fait classique, ci

## Exercices

Quatre exercices, à faire tourner dans les cellules qui suivent ; chacune contient sa valeur attendue. Ils s'appuient sur la boîte à outils construite plus haut : `d_lambda_produit`, `d_lambda_ferme`, `d_lambda_weyl`, `witt_nombre`, `serie_hilbert_libre`, `chebyshev_rescale`, `dim_sp`, `bornes`, `mu_mobius`.

In [11]:
def somme_witt(g: int, N: int) -> int:
    """TODO etudiant : somme des dimensions des pieces de degre 1 a N de l'algebre de Lie
    libre sur 2g generateurs. C'est le majorant de la dimension locale dim(U^dR_N / F^0)
    de la Proposition 4.15 (au terme g^N pres).

    Attendu sur g=2, N=4 : 4 + 6 + 20 + 60 = 90.
    """
    pass
    # TODO etudiant
    resultat = None  # TODO etudiant

In [12]:
def table_bornes(g: int, poids: list) -> list:
    """TODO etudiant : pour chaque poids dominant lam de la liste, renvoyer un dictionnaire
    {'lambda': lam, 'd': d_lambda, 'borne': min(triviale, non triviale), 'critere': bool}.
    Utiliser les fonctions de la boite a outils (d_lambda_produit, dim_sp, norme_lambda).

    Attendu sur g=2 et lam=(2, 2) : d = 14, et la borne retenue est la borne triviale.
    """
    pass
    # TODO etudiant
    resultat = None  # TODO etudiant

In [13]:
def points_selmer(p: int) -> int:
    """TODO etudiant : nombre de points projectifs F_p de la cubique de Selmer
    3x^3 + 4y^3 + 5z^3 = 0.
    Penser au passage du compte affine (triples) au compte projectif : chaque point
    projectif non nul admet (p - 1) representants affines.

    Attendu : points_selmer(7) = 3, points_selmer(11) = 12.
    """
    pass
    # TODO etudiant
    resultat = None  # TODO etudiant

In [14]:
def trace_parite(n: int) -> list:
    """TODO etudiant : les traces tr(sigma | A_k) pour k = 0, ..., n, obtenues par la
    recurrence tr(sigma | A_k) = tr(sigma | A_{k-2}), avec tr(sigma | A_0) = 1.
    Ces traces sont nulles sur les degres impairs, et c'est la que se joue le 1/2
    de la Remark 4.17.

    Attendu sur n = 5 : [1, 0, 1, 0, 1, 0].
    """
    pass
    # TODO etudiant
    resultat = None  # TODO etudiant

## Bilan

### Exécuté dans ce carnet

- la structure de racines de C_g, et ρ = (g, …, 1) ;
- d_λ par le produit de la Remark 3.2, par sa forme close, et par la sommation sur le groupe de Weyl — trois routes concordantes ;
- D = 2g²+g, la table des deux bornes du théorème 1.1 et le critère de la Remark 3.2 ;
- les nombres de Witt, la série de Hilbert de l'algebre de Lie libre et son terme quadratique en ½ ;
- les polynômes de Chebyshev rescalés, vérifiés numériquement contre P_d(2 cos θ) = 2 cos(dθ) et contre la forme close ;
- le mécanisme de parité qui donne dim L_k⁻ = b_k/2 pour k impair ;
- le bilan arithmétique de la condition de Chabauty–Kim, et le mécanisme du gain de puissance ;
- le cas n = 1, et les points de la cubique de Selmer sur les petits corps finis.

### Cité, non reproduit

La théorie d'Iwasawa non commutative, les annulations de sommes de caractères (§4), l'estimation de second moment (Proposition 4.14), l'intégration de Coleman. La formalisation Lean de l'énoncé est **hors de portée aujourd'hui** : Mathlib fournit `SymplecticGroup` et quelques briques de théorie des nombres p-adiques, mais **ni cohomologie galoisienne de G_Q, ni π₁ unipotent, ni Hodge p-adique**.

### Où ce carnet s'insère

Ce carnet prolonge [05 — Szpiro–Pasten](05-szpiro-pasten-2026.ipynb) — une autre conjecture de dimension, côté conducteurs — et s'insère dans le parcours ouvert par [01 — Formes modulaires et opérateurs de Hecke](01-formes-modulaires-sl2z-hecke.ipynb) ; le groupe symplectique et ses représentations sont calculés dans [Serre100/25 — le groupe Galois calculé](../Serre100/25-groupe-galois-calcule.ipynb).

**Série Langlands** | Précédent : [06 — Les opérateurs de Hecke au noyau](06-operateurs-de-hecke-natif.ipynb) | ↑ Série : [README de la série](README.md)

### Référence

Sa'ar Zehavi (MPI for Mathematics), *The Chabauty–Kim dimension inequality via noncommutative Iwasawa theory*, arXiv:2611.18218v1 [math.NT], 13 novembre 2026, 61 p. — **préprint de trois jours, non relu**.

Copie archivée (hors dépôt) : `G:\Mon Drive\MyIA\IA\Bibliographie IA\NumberTheory\2026 - Zehavi - The Chabauty-Kim dimension inequality via noncommutative Iwasawa theory (arXiv 2611.18218v1).pdf`